# Code to determine the Correlations metric


In [59]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from pathlib import Path
# utilized in upset_frequency
def load_all_league_data():
    """
    Load match data and standings for all available leagues.
    
    Returns:
        tuple: (league_data_dict, standings_dict) where each is a dict of league_name -> DataFrame
    """
    base_dir = Path().resolve()
    
    # Define league configurations
    leagues = {
        'Premier League': {
            'matches_file': 'premier_league1_std.csv',
            'standings_file': 'premier_league_standings_all_seasons.csv'
        },
        'Serie A': {
            'matches_file': 'serie_a_std.csv',
            'standings_file': 'serie_a_standings_all_seasons.csv'
        },
        'Bundesliga': {
            'matches_file': 'bundesliga_std.csv',
            'standings_file': 'bundesliga_standings_all_seasons.csv'
        },
        'La Liga': {
            'matches_file': 'la_liga_std.csv',
            'standings_file': 'la_liga_standings_all_seasons.csv'
        }
    }
    
    league_data = {}
    standings_data = {}
    
    print("Loading league data...")   
    for league_name, config in leagues.items():
        # Load match data
        matches_path = base_dir / "../csv/processed" / config['matches_file']
        if matches_path.exists():
            matches_df = pd.read_csv(matches_path)
            matches_df['date'] = pd.to_datetime(matches_df['date'])
            league_data[league_name] = matches_df
            print(f"✓ Loaded {league_name} matches: {len(matches_df):,} matches")
        else:
            print(f"✗ Warning: {matches_path} not found")
            
        # Load standings data
        standings_path = base_dir / "../csv/auxiliary" / config['standings_file']
        if standings_path.exists():
            standings_df = pd.read_csv(standings_path)
            standings_data[league_name] = standings_df
            print(f"✓ Loaded {league_name} standings: {len(standings_df):,} team-seasons")
        else:
            print(f"✗ Warning: {standings_path} not found")
    
    print(f"\nSuccessfully loaded {len(league_data)} leagues")
    return league_data, standings_data
league_data, standings_data = load_all_league_data()


Loading league data...
✓ Loaded Premier League matches: 8,360 matches
✓ Loaded Premier League standings: 440 team-seasons
✓ Loaded Serie A matches: 8,518 matches
✓ Loaded Serie A standings: 454 team-seasons
✓ Loaded Bundesliga matches: 6,426 matches
✓ Loaded Bundesliga standings: 378 team-seasons
✓ Loaded La Liga matches: 8,740 matches
✓ Loaded La Liga standings: 460 team-seasons

Successfully loaded 4 leagues


In [ ]:

results = []
def goal_calculation(team, games):
    home_goals = games.loc[games["home_team"] == team, "home_team_points"].sum() #sum up every home games goals
    away_goals = games.loc[games["away_team"] == team, "away_team_points"].sum() #sum up every away games goals
    return home_goals + away_goals


def correlation(team,season, matches_played):
    df = matches_played[matches_played["season"] == season]
    df = df[(df["home_team"] == team) | (df["away_team"] == team)]
    df = df.sort_values(by = "date", ascending= True)
    midpoint = len(df) // 2
    first_half = df.iloc[:midpoint]
    second_half = df.iloc[midpoint:]
    first_half_goals = goal_calculation(team, first_half)
    second_half_goals = goal_calculation(team, second_half)
    dictionary= {"team": team, "season": season, "first_half_goals": first_half_goals, "second_half_goals": second_half_goals}

    return dictionary


for league in league_data:
    df = league_data[league]
    unique_pairs = set(zip(df["season"], df["home_team"]))
    for season, team in unique_pairs:
        second=correlation(team, season, df)
        second["league"] = league
        results.append(second)
allleagues = pd.DataFrame(results)
allleagues=allleagues.sort_values(by = ["league", "season", "team"], ascending= True)

Loading league data...
✓ Loaded Premier League matches: 8,360 matches
✓ Loaded Premier League standings: 440 team-seasons
✓ Loaded Serie A matches: 8,518 matches
✓ Loaded Serie A standings: 454 team-seasons
✓ Loaded Bundesliga matches: 6,426 matches
✓ Loaded Bundesliga standings: 378 team-seasons
✓ Loaded La Liga matches: 8,740 matches
✓ Loaded La Liga standings: 460 team-seasons

Successfully loaded 4 leagues


In [ ]:
allleagues=allleagues.sort_values(by = ["league", "season", "team"], ascending= True)
allleagues.to_csv("correlation_metrics.csv", index=False)

# Create Plots

In [ ]:
allleagues = pd.read_csv("/Users/joeyli/skillvsluck/data/european_soccer/csv/analysis/correlation_metrics.csv")

ax = None

for league in combined_seasonal.groupby("league"):
    ax = df.plot(
        x="season", y="upset_frequency", kind="line", ax=ax, label=league, marker='o',          # circle marker
        linestyle='-',       # line between points
        markersize=6,        # size of dots
        linewidth=2

    )

plt.title("Upset Frequency by Season (Betting)")
plt.xlabel("Season")
plt.ylabel("Upset Frequency")
plt.legend(title="League")
plt.tight_layout()
plt.savefig("betting_combined_line_upset_freq.png")

AttributeError: 'dict' object has no attribute 'groupby'